The next step would be to run some very basic test cases and thereby explore how to manage the code and compare to other codes. The first setting would be a terrain without slope and only one type of vegetation and no wind. Then the wildfire should spread from an initial cell in a circular way. Likewise elliptic spread could be realized. 

In [1]:
import argparse
import datetime
import timeit

import torch

from pytorchfire.model import WildfireModel

# Experiment 1: no slope, no wind, 1 type of vegetation


In [3]:
# Run Experiment 1: no slope, no wind, 1 type of vegetation

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Simulation settings
SIZE = 200
NUM_STEPS = 150

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Environment: flat terrain, single vegetation type, no wind
p_veg = torch.ones(SIZE, SIZE) * 0.5          # uniform vegetation factor
p_den = torch.zeros(SIZE, SIZE)               # no density variation
wind_velocity = torch.zeros(SIZE, SIZE)       # no wind
wind_towards_direction = torch.zeros(SIZE, SIZE)

# slope: all zeros (flat terrain), with 3x3 neighborhood per cell
slope = torch.zeros(SIZE, SIZE, 3, 3)

# initial ignition: single cell in the center
initial_ignition = torch.zeros(SIZE, SIZE, dtype=torch.bool)
center = SIZE // 2
initial_ignition[center, center] = True

env_data = {
    'p_veg': p_veg,
    'p_den': p_den,
    'wind_velocity': wind_velocity,
    'wind_towards_direction': wind_towards_direction,
    'slope': slope,
    'initial_ignition': initial_ignition,
}

# Model parameters (no slope / wind effects)
params = {
    'a': torch.tensor(0.0),        # slope factor off
    'p_h': torch.tensor(0.35),     # base spread probability
    'p_continue': torch.tensor(0.3),
    'c_1': torch.tensor(0.0),      # wind magnitude effect off
    'c_2': torch.tensor(0.0),      # wind direction effect off
}

model = WildfireModel(env_data=env_data, params=params).to(device)
model.eval()
model.reset()

# Run simulation and record affected cells (burning OR burned)
frames = []
for step in range(NUM_STEPS):
    burning, burned = model.state
    affected = (burning | burned).detach().cpu().numpy()
    frames.append(affected)
    model.compute()

# Visualization: animated fire spread
fig, ax = plt.subplots(figsize=(5, 5))
img = ax.imshow(frames[0], cmap="hot", interpolation="nearest", vmin=0, vmax=1)
ax.set_title("Wildfire spread (step 0)")
ax.axis("off")


def update(frame_idx):
    img.set_data(frames[frame_idx])
    ax.set_title(f"Wildfire spread (step {frame_idx})")
    return [img]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=100,
    blit=True,
)

# Close the static figure and display JS-based animation instead
plt.close(fig)
HTML(ani.to_jshtml())